In [ ]:
# Import necessary libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline

def calc_parc_o3(var, p_top=30, p_bottom=100):
    """
    Calculate the partial ozone column between specified pressure levels.
    
    Parameters:
        var: xarray.DataArray, ozone data
        p_top: float, top pressure level (hPa)
        p_bottom: float, bottom pressure level (hPa)
    
    Returns:
        xarray.DataArray: Partial ozone column (in DU) with zeros removed
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    # Determine the pressure level coordinate name
    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    elif 'level' in var.dims:
        plev = var.level
        level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # Determine conversion factors based on the ordering and units of plev
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100
        factor_2 = 1  # conversion Pa->hPa
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1
        factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100

    # Calculate partial column for increasing pressure levels
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16  # Convert to DU
    else:
        for i in range(0, len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)

def spatial_average(O3_data, lat_start=60, lat_end=90):
    """
    Compute the spatial (latitude) average of the ozone data.
    
    Parameters:
        O3_data: xarray.DataArray, input ozone data
        lat_start: float, starting latitude
        lat_end: float, ending latitude
    
    Returns:
        xarray.DataArray: spatially averaged data
    """
    O3_avg = O3_data.mean(dim='lon')
    var = O3_avg.sel(lat=slice(lat_start, lat_end))
    weights = np.cos(np.deg2rad(var.lat))
    var = var.weighted(weights).mean(dim='lat')
    return var

def load_experiment_data(file_pattern):
    """
    Load and process ozone data from the given file pattern.
    
    The function reads multiple NetCDF files, computes the 30-70 hPa partial ozone column,
    applies spatial averaging over 60°-90° latitude using cosine weighting, and returns the result.
    
    Parameters:
        file_pattern: str, path pattern to the NetCDF files
    
    Returns:
        xarray.DataArray: processed ozone data
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    o3 = ds['O3'].mean(dim='lon')
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    var = o3_30_70.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(var.lat))
    var = var.weighted(weights).mean(dim='lat')
    return var

def load_reference_data(year):
    """
    Load the reference and climatology data for the specified year from the new dataset.
    
    New dataset:
      - Files are located in: /mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/O3
      - File naming: CO2x1SmidEmin_yBWCN.cam.h1.YYYY.O3.isobar.nc, where YYYY: first two digits represent ensemble member, last two digits represent year.
      - Each file represents one year of 4D ozone (O3) data with dimensions: time (360 timesteps, no leap calendar), lev (23 levels), lat (96), lon (144),
        along with latitude, longitude coordinates, Gaussian weights, and date information.
    
    This function merges these files for a total of 200 years, computes the partial ozone column (30-70 hPa),
    applies spatial averaging over 60°-90° latitude, then filters reference data for the given year and computes the climatology.
    
    Parameters:
        year: str, e.g., '0008'
    
    Returns:
        tuple: (ref_data, clim_data) both as xarray.DataArray
    """
    file_pattern = '/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/O3/CO2x1SmidEmin_yBWCN.cam.h1.*.O3.isobar.nc'
    ds = xr.open_mfdataset(file_pattern, combine='by_coords')
    o3 = ds['O3']
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    var_all = spatial_average(o3_30_70, lat_start=60, lat_end=90)
    
    # Attempt to select reference data for the given year
    #ref_year = int(year) + 100  # 模拟年 8 对应年份 108
    #ref_data = var_all.sel(time=var_all.time.dt.year == ref_year)
#    ref_data = var_all.sel(time=var_all.time.dt.year == int(year))
    
    # If no data is found, fall back to previous merged dataset
    '''
    if ref_data.size == 0:
        print(f"No new reference data for year {year}. Falling back to previous merged dataset.")
        merged_file = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
        ds_old = xr.open_dataset(merged_file)
        o3_old = ds_old['O3']
        o3_30_70_old = calc_parc_o3(o3_old, 30, 70)
        var_all_old = spatial_average(o3_30_70_old, lat_start=60, lat_end=90)
        ref_data = var_all_old.sel(time=var_all_old.time.dt.year == int(year))
        clim_data = var_all_old.groupby('time.dayofyear').mean()
        return ref_data, clim_data
    '''
    merged_file = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
    ds_old = xr.open_dataset(merged_file)
    o3_old = ds_old['O3']
    o3_30_70_old = calc_parc_o3(o3_old, 30, 70)
    var_all_old = spatial_average(o3_30_70_old, lat_start=60, lat_end=90)
    ref_data = var_all_old.sel(time=var_all_old.time.dt.year == int(year))
    clim_data = var_all.groupby('time.dayofyear').mean()
    return ref_data, clim_data

def load_year_small_pert(year):
    """
    Load February and March small perturbation data for a given year.
    """
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base_path, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base_path, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    data_Feb = load_experiment_data(file_Feb)
    data_Mar = load_experiment_data(file_Mar)
    return data_Feb, data_Mar

def load_year_nocouple(year):
    """
    Load February and March nocouple data for a given year from the nocouple_data directory.
    """
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    data_Feb = load_experiment_data(file_Feb)
    data_Mar = load_experiment_data(file_Mar)
    return data_Feb, data_Mar

def unified_plot(ref_data, clim_data, experiments, year, pressure_range='30-70',
                 output_filename_prefix='O3_evolution',
                 xlim_start=0, xlim_end=150,
                 xticks_default=None, xtick_labels_default=None):
    """
    Plot the evolution of the partial ozone column with reference and climatology curves,
    and also plot the average line for each experiment.
    
    Parameters:
        ref_data: xarray.DataArray for the reference curve (assumed without a member dimension)
        clim_data: xarray.DataArray for the climatology curve (assumed without a member dimension)
        experiments: list of dictionaries, each with keys:
            'data': xarray.DataArray (may have a 'member' dimension)
            'label': string label for the experiment
            'offset': integer value indicating the starting x-axis offset (in days) for the data
            'line_color': color for the line (if applicable)
            'fill_color': color for the envelope shading (if data has a 'member' dimension)
        year: string, e.g., '0008', used for labeling the plot
        pressure_range: string, used in the y-axis label
        output_filename_prefix: string prefix for saving the plot files
        xlim_start: starting x-axis value (in days, e.g., 0 for Jan 1)
        xlim_end: ending x-axis value (in days)
        xticks_default: list of x-tick positions; if None, determined from full 12-month info
        xtick_labels_default: list of corresponding x-tick labels; if None, determined from full 12-month names
    
    Returns:
        tuple: (fig, ax) of the generated plot
    """
    # Full month names and corresponding approximate start days (days from Jan 1, starting at 0)
    all_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    all_month_ticks = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

    if xticks_default is None:
        xticks_default = [tick for tick in all_month_ticks if xlim_start <= tick <= xlim_end]
        if not xticks_default:
            xticks_default = [xlim_start, xlim_end]
    if xtick_labels_default is None:
        xtick_labels_default = [all_months[i] for i, tick in enumerate(all_month_ticks) if tick in xticks_default]

    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

    # Plot reference data
    if 'time' in ref_data.coords:
        total = ref_data.sizes['time']
        start_idx = int(xlim_start) if xlim_start < total else total
        end_idx = int(min(xlim_end, total))
        effective_ref = ref_data.isel(time=slice(start_idx, end_idx))
        x_ref = np.arange(start_idx, end_idx)
    else:
        total = len(ref_data)
        start_idx = int(xlim_start) if xlim_start < total else total
        end_idx = int(min(xlim_end, total))
        effective_ref = ref_data[start_idx:end_idx]
        x_ref = np.arange(start_idx, end_idx)
    ax.plot(x_ref, effective_ref, color='black', linewidth=5, label='Reference')

    # Plot climatology data
    if 'time' in clim_data.coords:
        total = clim_data.sizes['time']
        start_idx = int(xlim_start) if xlim_start < total else total
        end_idx = int(min(xlim_end, total))
        effective_clim = clim_data.isel(time=slice(start_idx, end_idx))
        x_clim = np.arange(start_idx, end_idx)
    else:
        total = len(clim_data)
        start_idx = int(xlim_start) if xlim_start < total else total
        end_idx = int(min(xlim_end, total))
        effective_clim = clim_data[start_idx:end_idx]
        x_clim = np.arange(start_idx, end_idx)
    ax.plot(x_clim, effective_clim, color='black', linestyle=':', linewidth=5, label='Climatology')

    # Plot experiment data
    for exp in experiments:
        data = exp['data']
        offset = exp['offset']
        label = exp['label']
        line_color = exp.get('line_color', 'gray')
        fill_color = exp.get('fill_color', 'gray')

        if 'time' in data.coords:
            total = data.sizes['time']
        else:
            total = data.shape[0]

        if offset >= xlim_end:
            continue

        if offset < xlim_start:
            start_idx = int(xlim_start - offset)
        else:
            start_idx = 0
        end_idx = int(min(total, xlim_end - offset))
        if end_idx <= start_idx:
            continue

        if 'time' in data.coords:
            effective_data = data.isel(time=slice(start_idx, end_idx))
        else:
            effective_data = data[start_idx:end_idx]
        x = np.arange(offset + start_idx, offset + end_idx)

        if 'member' in effective_data.dims:
            # 先保证 member 在第一个维度，便于逐条画线
            if effective_data.dims[0] != 'member':
                plot_data = effective_data.transpose('member', ...)
            else:
                plot_data = effective_data

            # 计算 ensemble mean 和 ±1 sigma
            avg_line = plot_data.mean(dim='member')
            std_line = plot_data.std(dim='member')
            upper = avg_line + std_line
            lower = avg_line - std_line

            # 画每一条 member 线：比平均线细一半
            mean_lw = 5
            member_lw = mean_lw / 2

            for i in range(plot_data.sizes['member']):
                member_line = plot_data.isel(member=i)
                ax.plot(
                    x, member_line,
                    color=line_color,
                    linewidth=member_lw,
                    alpha=0.5
                )

            # 画 ±1 sigma 包络区
            ax.fill_between(
                x, lower, upper,
                color=fill_color,
                alpha=0.25,
                label=f'{label} ±1σ'
            )

            # 画平均线
            ax.plot(
                x, avg_line,
                color=line_color,
                linewidth=mean_lw,
                label=f'{label} mean'
            )

        else:
            ax.plot(x, effective_data, color=line_color, linewidth=5, label=label)

    ax.set_xticks(xticks_default)
    ax.set_xticklabels(xtick_labels_default, fontsize=16)
    ax.set_xlim(xlim_start, xlim_end)
    ax.set_ylabel(f'Partial ozone column, {pressure_range} hPa (DU)', fontsize=18)
    ax.set_ylim(70, 160)
    ax.tick_params(axis='y', labelsize=16)

    ax.text(0.02, 0.95, f'Year: {year}', transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.legend(fontsize=14)

    # Updated output folder for saving plots
    output_folder = '/home/weiji/restart_exam/20250221/O3withnewclim'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    out_pdf = f'{output_folder}/{output_filename_prefix}_{year}_{pressure_range.replace("-", "to")}.pdf'
    out_png = f'{output_folder}/{output_filename_prefix}_{year}_{pressure_range.replace("-", "to")}.png'
    fig.savefig(out_pdf)
    fig.savefig(out_png)

    return fig, ax

# --- Data Loading and Plotting ---

# 0008 Small Perturbation Data
file_0008_Jan_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'

data_0008_Jan_small = load_experiment_data(file_0008_Jan_small)
data_0008_Feb_small = load_experiment_data(file_0008_Feb_small)
data_0008_Mar_small = load_experiment_data(file_0008_Mar_small)

# 0008 Large Perturbation Data
file_0008_Feb_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Apr_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'

data_0008_Feb_large = load_experiment_data(file_0008_Feb_large)
data_0008_Mar_large = load_experiment_data(file_0008_Mar_large)
data_0008_Apr_large = load_experiment_data(file_0008_Apr_large)

# 0008 Moist (Vapor) Perturbation Data
file_0008_Feb_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'

data_0008_Feb_moist = load_experiment_data(file_0008_Feb_moist)
data_0008_Mar_moist = load_experiment_data(file_0008_Mar_moist)

# 0008 Nocouple Data: FEB from FNC folder (to be marked as reference) and MAR from nocouple_data
file_0008_Feb_nocouple_ref = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc'
data_0008_Feb_nocouple_ref = load_experiment_data(file_0008_Feb_nocouple_ref)

data_0008_Feb_nocouple, data_0008_Mar_nocouple = load_year_nocouple('0008')

# Load new reference and climatology data for 0008 from the updated dataset,
# with fallback to previous merged data if necessary.
ref_0008, clim_0008 = load_reference_data('0008')

### Figure 1: 0008Jan small pert, 0008Feb small pert, 0008Mar small pert
experiments_fig1 = [
    {'data': data_0008_Jan_small, 'label': '0008Jan_small_pert', 'offset': 0, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Feb_small, 'label': '0008Feb_small_pert', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'},
    {'data': data_0008_Mar_small, 'label': '0008Mar_small_pert', 'offset': 59, 'line_color': 'hotpink', 'fill_color': 'hotpink'}
]

fig1, ax1 = unified_plot(ref_0008, clim_0008, experiments_fig1, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig1')
plt.show()

### Figure 2: 0008Feb large pert, 0008Mar large pert, 0008Apr large pert
experiments_fig2 = [
    {'data': data_0008_Feb_large, 'label': '0008Feb_large_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Mar_large, 'label': '0008Mar_large_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'},
    {'data': data_0008_Apr_large, 'label': '0008Apr_large_pert', 'offset': 90, 'line_color': 'hotpink', 'fill_color': 'hotpink'}
]

fig2, ax2 = unified_plot(ref_0008, clim_0008, experiments_fig2, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig2')
plt.show()

### Figure 3: 0008Feb small pert, 0008Feb large pert, 0008Feb moist pert
experiments_fig3 = [
    {'data': data_0008_Feb_small, 'label': '0008Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Feb_large, 'label': '0008Feb_large_pert', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'},
    {'data': data_0008_Feb_moist, 'label': '0008Feb_moist_pert', 'offset': 31, 'line_color': 'hotpink', 'fill_color': 'hotpink'}
]

fig3, ax3 = unified_plot(ref_0008, clim_0008, experiments_fig3, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig3')
plt.show()

### Figure 4: 0008Mar small pert, 0008Mar large pert, 0008Mar moist pert
experiments_fig4 = [
    {'data': data_0008_Mar_small, 'label': '0008Mar_small_pert', 'offset': 59, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Mar_large, 'label': '0008Mar_large_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'},
    {'data': data_0008_Mar_moist, 'label': '0008Mar_moist_pert', 'offset': 59, 'line_color': 'hotpink', 'fill_color': 'hotpink'}
]

fig4, ax4 = unified_plot(ref_0008, clim_0008, experiments_fig4, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig4')
plt.show()

# Figure 5: Year 0003 small pert (Feb and Mar)
data_0003_Feb_small, data_0003_Mar_small = load_year_small_pert('0003')
ref_0003, clim_0003 = load_reference_data('0003')

experiments_fig5 = [
    {'data': data_0003_Feb_small, 'label': '0003Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0003_Mar_small, 'label': '0003Mar_small_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig5, ax5 = unified_plot(ref_0003, clim_0003, experiments_fig5, year='0003', pressure_range='30-70', output_filename_prefix='O3_evolution_fig5')
plt.show()

# Figure 6: Year 0013 small pert (Feb and Mar)
data_0013_Feb_small, data_0013_Mar_small = load_year_small_pert('0013')
ref_0013, clim_0013 = load_reference_data('0013')

experiments_fig6 = [
    {'data': data_0013_Feb_small, 'label': '0013Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0013_Mar_small, 'label': '0013Mar_small_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig6, ax6 = unified_plot(ref_0013, clim_0013, experiments_fig6, year='0013', pressure_range='30-70', output_filename_prefix='O3_evolution_fig6')
plt.show()

# Figure 7: Year 0014 small pert (Feb and Mar)
data_0014_Feb_small, data_0014_Mar_small = load_year_small_pert('0014')
ref_0014, clim_0014 = load_reference_data('0014')

experiments_fig7 = [
    {'data': data_0014_Feb_small, 'label': '0014Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0014_Mar_small, 'label': '0014Mar_small_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig7, ax7 = unified_plot(ref_0014, clim_0014, experiments_fig7, year='0014', pressure_range='30-70', output_filename_prefix='O3_evolution_fig7')
plt.show()

# Figure 8: Year 0019 small pert (Feb and Mar)
data_0019_Feb_small, data_0019_Mar_small = load_year_small_pert('0019')
ref_0019, clim_0019 = load_reference_data('0019')

experiments_fig8 = [
    {'data': data_0019_Feb_small, 'label': '0019Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0019_Mar_small, 'label': '0019Mar_small_pert', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig8, ax8 = unified_plot(ref_0019, clim_0019, experiments_fig8, year='0019', pressure_range='30-70', output_filename_prefix='O3_evolution_fig8')
plt.show()

# Figure 9: 0008 Nocouple Data: FEB (from FNC reference) and MAR
experiments_fig9 = [
    {'data': data_0008_Feb_nocouple_ref, 'label': '0008Feb_nocouple', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Mar_nocouple, 'label': '0008Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig9, ax9 = unified_plot(ref_0008, clim_0008, experiments_fig9, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig9')
plt.show()

# Figure 10: 0013 Nocouple Data (Feb and Mar)
data_0013_Feb_nocouple, data_0013_Mar_nocouple = load_year_nocouple('0013')
experiments_fig10 = [
    {'data': data_0013_Feb_nocouple, 'label': '0013Feb_nocouple', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0013_Mar_nocouple, 'label': '0013Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig10, ax10 = unified_plot(ref_0013, clim_0013, experiments_fig10, year='0013', pressure_range='30-70', output_filename_prefix='O3_evolution_fig10')
plt.show()

# Figure 11: 0014 Nocouple Data (Feb and Mar)
data_0014_Feb_nocouple, data_0014_Mar_nocouple = load_year_nocouple('0014')
experiments_fig11 = [
    {'data': data_0014_Feb_nocouple, 'label': '0014Feb_nocouple', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0014_Mar_nocouple, 'label': '0014Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig11, ax11 = unified_plot(ref_0014, clim_0014, experiments_fig11, year='0014', pressure_range='30-70', output_filename_prefix='O3_evolution_fig11')
plt.show()

# Figure 12: 0019 Nocouple Data (Feb and Mar)
data_0019_Feb_nocouple, data_0019_Mar_nocouple = load_year_nocouple('0019')
experiments_fig12 = [
    {'data': data_0019_Feb_nocouple, 'label': '0019Feb_nocouple', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0019_Mar_nocouple, 'label': '0019Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig12, ax12 = unified_plot(ref_0019, clim_0019, experiments_fig12, year='0019', pressure_range='30-70', output_filename_prefix='O3_evolution_fig12')
plt.show()

# Figure 13: 0008Feb small pert vs 0008Feb nocouple
experiments_fig13 = [
    {'data': data_0008_Feb_small, 'label': '0008Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Feb_nocouple_ref, 'label': '0008Feb_nocouple', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig13, ax13 = unified_plot(ref_0008, clim_0008, experiments_fig13, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig13')
plt.show()

# Figure 14: 0008Mar small pert vs 0008Mar nocouple
experiments_fig14 = [
    {'data': data_0008_Mar_small, 'label': '0008Mar_small_pert', 'offset': 59, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0008_Mar_nocouple, 'label': '0008Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig14, ax14 = unified_plot(ref_0008, clim_0008, experiments_fig14, year='0008', pressure_range='30-70', output_filename_prefix='O3_evolution_fig14')
plt.show()

# Figure 15: 0013Feb small pert vs 0013Feb nocouple
experiments_fig15 = [
    {'data': data_0013_Feb_small, 'label': '0013Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0013_Feb_nocouple, 'label': '0013Feb_nocouple', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig15, ax15 = unified_plot(ref_0013, clim_0013, experiments_fig15, year='0013', pressure_range='30-70', output_filename_prefix='O3_evolution_fig15')
plt.show()

# Figure 16: 0013Mar small pert vs 0013Mar nocouple
experiments_fig16 = [
    {'data': data_0013_Mar_small, 'label': '0013Mar_small_pert', 'offset': 59, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0013_Mar_nocouple, 'label': '0013Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig16, ax16 = unified_plot(ref_0013, clim_0013, experiments_fig16, year='0013', pressure_range='30-70', output_filename_prefix='O3_evolution_fig16')
plt.show()

# Figure 17: 0014Feb small pert vs 0014Feb nocouple
experiments_fig17 = [
    {'data': data_0014_Feb_small, 'label': '0014Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0014_Feb_nocouple, 'label': '0014Feb_nocouple', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig17, ax17 = unified_plot(ref_0014, clim_0014, experiments_fig17, year='0014', pressure_range='30-70', output_filename_prefix='O3_evolution_fig17')
plt.show()

# Figure 18: 0014Mar small pert vs 0014Mar nocouple
experiments_fig18 = [
    {'data': data_0014_Mar_small, 'label': '0014Mar_small_pert', 'offset': 59, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0014_Mar_nocouple, 'label': '0014Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig18, ax18 = unified_plot(ref_0014, clim_0014, experiments_fig18, year='0014', pressure_range='30-70', output_filename_prefix='O3_evolution_fig18')
plt.show()

# Figure 19: 0019Feb small pert vs 0019Feb nocouple
experiments_fig19 = [
    {'data': data_0019_Feb_small, 'label': '0019Feb_small_pert', 'offset': 31, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0019_Feb_nocouple, 'label': '0019Feb_nocouple', 'offset': 31, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig19, ax19 = unified_plot(ref_0019, clim_0019, experiments_fig19, year='0019', pressure_range='30-70', output_filename_prefix='O3_evolution_fig19')
plt.show()

# Figure 20: 0019Mar small pert vs 0019Mar nocouple
experiments_fig20 = [
    {'data': data_0019_Mar_small, 'label': '0019Mar_small_pert', 'offset': 59, 'line_color': 'forestgreen', 'fill_color': 'forestgreen'},
    {'data': data_0019_Mar_nocouple, 'label': '0019Mar_nocouple', 'offset': 59, 'line_color': 'royalblue', 'fill_color': 'royalblue'}
]
fig20, ax20 = unified_plot(ref_0019, clim_0019, experiments_fig20, year='0019', pressure_range='30-70', output_filename_prefix='O3_evolution_fig20')
plt.show()
